# Company research: load, inspect, validate, refresh

This notebook reads the included research snapshots. It does not claim that all companies, source conflicts, global comparables or forecasts are finished. The detailed bilingual segment pilot covers Wendy’s only.

**Viewing saved data requires no API credentials.** Downloading new WRDS/LSEG data requires authorized credentials and provider entitlements. Never put credentials into a notebook cell, output, repository, or browser code.


In [ ]:
from pathlib import Path
import csv, json, gzip, hashlib, subprocess, sys, os
import pandas as pd
csv.field_size_limit(2**31-1)
ROOT = Path.cwd()
if not (ROOT/'data/master/parts.json').exists():
    raise FileNotFoundError('Run this notebook from the repository root.')
manifest = pd.read_csv(ROOT/'file_manifest.csv')
display(manifest[['path','bytes']].head(15))
print('Packaged files:', len(manifest))
subprocess.run([sys.executable, str(ROOT/"tools/restore_transport.py")], check=True)


## 1. Restore the full master without loading 1.5 GB into memory

The gzip file is split into byte parts for Git transport. Reassembly verifies its SHA-256. Do not open the decompressed CSV in a spreadsheet; use the streaming reader below. The master includes cached evidence; smaller direct CSVs are easier for review.

In [ ]:
MASTER = ROOT/'outputs/company_research_master/Company_Research_Master.csv.gz'
if not MASTER.exists():
    subprocess.run([sys.executable, str(ROOT/'tools/restore_master.py')], check=True)
parts = json.loads((ROOT/'data/master/parts.json').read_text())
assert hashlib.sha256(MASTER.read_bytes()).hexdigest() == parts['combined_sha256']
print('Master bytes verified:', MASTER.stat().st_size)

In [ ]:
def records(dataset):
    with gzip.open(MASTER, 'rt', encoding='utf-8-sig', newline='') as f:
        for row in csv.DictReader(f):
            if row['dataset'] == dataset:
                yield json.loads(row['record_json'])

def read_table(relative_path):
    path = ROOT/relative_path
    if not path.exists() and path.with_suffix(path.suffix+'.gz').exists():
        path=path.with_suffix(path.suffix+'.gz')
    return pd.read_csv(path, low_memory=False)

results=read_table('outputs/company_research_master/Comparable_Results_Excel_View.csv')
print('Existing comparable-pair rows:',len(results))
display(results.head(10))

## 2. Wendy’s complete pilot and evidence

English/Chinese descriptions are research syntheses, not verbatim quotations. Source claims retain filing/publication dates, fact periods and URLs. A missing value is not a claim that the business attribute does not exist.

In [ ]:
pilot=read_table('outputs/wendys_segment_facets/Wendys_Company_and_Segment_Facets.csv')
assert len(pilot)==72
core=pilot[pilot['Record Type']=='CORE']
assert len(core)==30 and core['Value Text'].notna().sum()==28
assert core.loc[core.Facet=='Primary Products','Value Text'].notna().all()
display(pilot[['Record Type','Facet','Description English','Source Links','Document Provenance']].head(12))

## 3. Packaging and existing validation

Checksums verify file integrity, not factual truth. Existing validation checks data boundaries and score reconciliation. Cross-source textual agreement does not establish independent factual verification; unresolved conflicts remain visible and excluded where required.

In [ ]:
bad=[]
for row in manifest.to_dict('records'):
    path=ROOT/row['path']
    if not path.exists() or hashlib.sha256(path.read_bytes()).hexdigest()!=row['sha256']:
        bad.append(row['path'])
assert not bad, bad
report=json.loads((ROOT/'outputs/global_comps_560/repaired_facets_20261028/validation.json').read_text())
assert report['target_external_information_leakage'] is False
assert report['scores_and_weights_reconcile'] is True
print(json.dumps(report,indent=2))
print('Package integrity verified; this is not a blanket factual-validation claim.')

## 4. Fetch new data when explicitly needed

- **WRDS:** the sample below uses `WRDS_API_TOKEN` from the local process environment. Table access depends on subscription and entitlements. It fetches one bounded page; the historical collection script handles its own catalog pagination.
- **SEC:** public API, descriptive `SEC_USER_AGENT`, no token. Download current submissions and select filing form/date before fetching filing text.
- **LSEG:** `lseg.data` uses a desktop session with a running authorized Workspace instance, or a separately entitled platform session. An app key is not a standalone universal data credential. Configure your local SDK session outside version control. See the acquisition scripts for the original field lists and five-year window.
- **Capital IQ / instructor Excel:** supplied files are inputs; they do not constitute API access.

Functions below are defined but **not called automatically**, avoiding unexpected downloads or overwriting snapshots.

In [ ]:
import requests

def fetch_wrds_page(table='comp/company', limit=5):
    token=os.environ.get('WRDS_API_TOKEN')
    if not token: raise RuntimeError('Configure WRDS_API_TOKEN locally.')
    if table not in {'comp/company','comp/g_company'}: raise ValueError('Review table before adding it.')
    response=requests.get(f'https://wrds-api.wharton.upenn.edu/data/{table}/',
        params={'limit':min(limit,100)}, headers={'Authorization':'Token '+token}, timeout=60)
    response.raise_for_status()
    return response.json()

def fetch_sec_submissions(cik):
    ua=os.environ.get('SEC_USER_AGENT')
    if not ua: raise RuntimeError('Configure SEC_USER_AGENT with your contact information.')
    response=requests.get(f'https://data.sec.gov/submissions/CIK{int(cik):010d}.json',
        headers={'User-Agent':ua,'Accept-Encoding':'gzip, deflate'}, timeout=45)
    response.raise_for_status()
    return response.json()

def fetch_lseg_fields(instruments, fields):
    import lseg.data as ld
    ld.open_session()  # Uses your locally configured, authorized session.
    try:
        return ld.get_data(universe=instruments, fields=fields)
    finally:
        ld.close_session()


## 5. Historical pipeline and refresh boundaries

Acquisition scripts are retained under `outputs/lseg_downloads/` and `outputs/global_comps_560/`. They document field choices, source handling, extraction and scoring. Some depend on their original cached paths or optional model dependencies and are not promised to run end-to-end from this reduced delivery alone. Use the functions and snapshots above first; inspect prerequisites before rerunning historical stages.

The original instructor workbook at its earlier Desktop path is unavailable. Prepared 560-target tables are included and explicitly identified as prepared exports rather than an original workbook. No credential is included. The global candidate results concern the previous candidate pool, not an exhaustive completed search of all listed companies.